In [ ]:
import json
from collections.abc import Sequence
import requests
from bs4 import BeautifulSoup

from langchain_community.utilities import DuckDuckGoSearchAPIWrapper as ddg
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END

from typing import Final, List, Dict, Any, TypedDict, Optional

NUM_SEARCH_QUERIES: Final[int] = 3
NUM_SEARCH_RESULTS_PER_QUERY: Final[int] = 3
RESULT_TEXT_MAX_CHARACTERS: Final[int] = 10000
LOCAL_MODEL: Final[str] = 'gemma4:e4b'
TEMPERATURE: Final[float] = 0.7
WEB_TIMEOUT: Final[int] = 15
HEADERS: Final[dict] = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'en-US,en;q=0.9',
}    

In [ ]:
class LocalLLM:
    def __init__(
        self,
        model: str=LOCAL_MODEL,
        temperature: float=TEMPERATURE,
    ):
        self._model = model
        self._temperature = temperature
        
    def __call__(self):
        return ChatOllama(
            model=self._model,
            temperature=self._temperature,
            validate_model_on_init=True,
        )

class WebSearch:
    def __init__(self, timeout:int = WEB_TIMEOUT):
        self._timeout = timeout
        self._headers = HEADERS

    def search(
        self,
        web_query: str,
        num_results: int,
    ) -> Sequence[str]:
        return [
            result['link'] for result in ddg().results(
                web_query, num_results
            )
        ]
        
    def scrape(self, url: str) -> str:
        """Scrape a webpage."""
        try:
            response = requests.get(url, headers=self._headers, timeout=self._timeout)
            if response.status_code == 200:
                return BeautifulSoup(
                    response.text, 'html.parser'
                ).get_text(separator=' ', strip=True)
            else:
                return f"Failed to retrieve the webpage: Status code {response.status_code}"
        except Exception as e:
            print(e)
            return f"Failed to retrieve the webpage: {e}"

In [ ]:
class AssistantInfo(TypedDict):
    assistant_type: str
    assistant_instructions: str
    user_question: str

class SearchQuery(TypedDict):
    search_query: str
    user_question: str

class SearchResult(TypedDict):
    result_url: str
    search_query: str
    user_question: str

class SearchSummary(TypedDict):
    summary: str
    result_url: str
    user_question: str

class ResearchReport(TypedDict):
    report: str

# Graph state
class ResearchState(TypedDict):
    llm: LocalLLM
    ws: WebSearch
    user_question: str
    assistant_info: Optional[AssistantInfo]
    search_queries: Optional[List[SearchQuery]]
    search_results: Optional[List[SearchResult]]
    search_summaries: Optional[List[SearchSummary]]
    research_summary: Optional[str]
    final_report: Optional[str]
    used_fallback_search: Optional[bool]
    relevance_evaluation: Optional[Dict[str, Any]]
    should_regenerate_queries: Optional[bool]
    iteration_count: Optional[int]

In [ ]:
class Assistant:
    def __init__(self, state) -> None:
        self._llm = state['llm']()
        self._prompt = PromptTemplate.from_template(
            template=self._read_template()
        ).format(user_question=state['user_question'])
                
    def _read_template(self) -> str:
        return '''
        You are skilled at assigning a research question to the correct research assistant. 
        There are various research assistants available, each specialized in an area of expertise. 
        Each assistant is identified by a specific type. Each assistant has specific instructions to undertake the research.
        
        How to select the corraect assistant:
        You must select the relevant assistant depending on the topic of the question, which should match the area of expertise of the assistant.
        
        
        Here are some examples on how to return the correct assistant information, depending on the question asked:
        
        **Examples:**
        
        Question: "Should I invest in Apple stocks?"
        Response: 
        {{
            "assistant_type": "Financial analyst assistant",
            "assistant_instructions": "You are a seasoned finance analyst AI assistant. Your primary goal is to compose comprehensive, astute, impartial, and methodically arranged financial reports based on provided data and trends.",
            "user_question": {user_question}
        }}
        
        Question: "what are the most interesting sites in Tel Aviv?"
        Response: 
        {{
            "assistant_type": "Tour guide assistant",
            "assistant_instructions": "You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.",
            "user_question": "{user_question}"
        }}
        
        
        Question: "Is Messi a good soccer player?"
        Response: 
        {{
            "assistant_type": "Sport expert assistant",
            "assistant_instructions": "You are an experienced AI sport assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured sport reports on given sport personalities, or sport events, including factual details, statistics and insights.",
            "user_question": "{user_question}"
        }}
        
        Return the output as a JSON without the ```json header.
        Now that you have understood all the above, select the correct reserach assistant for the following question.
        Question: {user_question}
        Response:
        '''

    def __call__(self) -> dict:    
        return {'assistant_info': json.loads(self._llm.invoke(self._prompt).content)}


In [ ]:
state = {
    'llm': LocalLLM(),
    'user_question': 'What can I see and do in the Spanish town of Astorga?'
}
result = Assistant(state)
print(result()['assistant_info']['assistant_instructions'])

In [ ]:
class GenerateSearchQueries:
    def __init__(self, state: Dict[str, Any]):
        self._llm = state['llm']()
        self._assistant_info = state["assistant_info"]
        self._user_question = state["user_question"]
        self._iteration_count = state.get("iteration_count", 0)
        self._previous_queries = state.get("search_queries", [])
        self._relevance_evaluation = state.get("relevance_evaluation", None)
    
    def _get_first_template(self):        
        WEB_SEARCH_INSTRUCTIONS = """
        {assistant_instructions}
        
        Write {num_search_queries} web search queries to gather as much information as possible 
        on the following question: {user_question}. Your objective is to write a report based on the information you find.
        You must respond with a list of queries such as query1, query2, query3 in the following format: 
        [
            {{"search_query": "query1", "user_question": "{user_question}" }},
            {{"search_query": "query2", "user_question": "{user_question}" }},
            {{"search_query": "query3", "user_question": "{user_question}" }}
        ]
        """        
        return PromptTemplate.from_template(template=WEB_SEARCH_INSTRUCTIONS)

    def _get_second_template(self):
        WEB_SEARCH_INSTRUCTIONS = '''
        {assistant_instructions}

        You are generating new search queries because the previous queries did not yield sufficiently relevant results.
        
        Original question: {user_question}
        
        Previous search queries: {previous_query_list}
        
        Relevance evaluation: {relevance_percentage}% relevant
        Explanation: {relevance_explanation}
        
        Please generate {num_search_queries} NEW and DIFFERENT web search queries that are MORE SPECIFIC and TARGETED 
        to gather relevant information on the original question. 
        
        IMPORTANT: DO NOT repeat or rephrase the previous queries. Create completely different approaches to finding information.
        
        You must respond with a list of queries in the following format:
        [
            {{"search_query": "query1", "user_question": "{user_question}" }},
            {{"search_query": "query2", "user_question": "{user_question}" }},
            {{"search_query": "query3", "user_question": "{user_question}" }}
        ]
        '''
        return PromptTemplate.from_template(template=WEB_SEARCH_INSTRUCTIONS)

    def _get_third_template(self):
        WEB_SEARCH_INSTRUCTIONS = '''
        {assistant_instructions}

        You are generating search queries for the FINAL attempt to find relevant information.
        
        Original question: {user_question}
        
        All previous search queries that DID NOT yield relevant results: {all_previous_queries}
        
        For this final attempt, take a completely different angle. Consider:
        1. Breaking down the question into smaller, more focused sub-questions
        2. Using technical or specialized terms related to the topic
        3. Searching for expert opinions or academic perspectives
        4. Looking for case studies or specific examples
        5. Exploring historical context or background information
        
        CRITICAL INSTRUCTIONS:
        1. DO NOT repeat or rephrase ANY previous queries listed above
        2. Generate queries that are COMPLETELY DIFFERENT from all previous attempts
        
        Please generate {num_search_queries} COMPLETELY NEW search queries following the strategy above.
        
        You must respond with a list of queries in the following format:
        [
            {{"search_query": "query1", "user_question": "{user_question}" }},
            {{"search_query": "query2", "user_question": "{user_question}" }},
            {{"search_query": "query3", "user_question": "{user_question}" }}
        ]
        '''
        return PromptTemplate.from_template(template=WEB_SEARCH_INSTRUCTIONS)

    def _get_prompt(self):
        if self._iteration_count == 0:
            # First-time query generation
            print("Generating initial search queries...")
            return self._get_first_template().format(
                assistant_instructions=self._assistant_info["assistant_instructions"],
                user_question=self._user_question,
                num_search_queries=NUM_SEARCH_QUERIES
            )
        elif self._iteration_count == 1:
            # Second iteration - more specific queries
            print("First regeneration: Creating more specific queries...")
            previous_query_list = ", ".join([q["search_query"] for q in self._previous_queries])
            relevance_percentage = self._relevance_evaluation.get("relevance_percentage", 0) if self._relevance_evaluation else 0
            relevance_explanation = self._relevance_evaluation.get("explanation", "No explanation provided") if self._relevance_evaluation else ""
            return self._get_second_template().format(
                assistant_instructions=self._assistant_info["assistant_instructions"],
                user_question=self._user_question,
                previous_query_list=previous_query_list,
                relevance_percentage=relevance_percentage,
                relevance_explanation=relevance_explanation,
                num_search_queries=NUM_SEARCH_QUERIES,
            )
                
        else:
            # Third or later iteration - completely different approach
            print(f"Iteration {self._iteration_count}: Using alternative search strategies...")
            all_previous_queries = ", ".join([q["search_query"] for q in self._previous_queries])
            return self._get_third_template().format(
                assistant_instructions=self._assistant_info["assistant_instructions"],
                user_question=self._user_question,
                all_previous_queries=all_previous_queries,
                num_search_queries=NUM_SEARCH_QUERIES,
            )


    def __call__(self):        
        prompt = self._get_prompt()
        response = self._llm.invoke(prompt)
        return {
            "search_queries": json.loads(response.content),
            "relevance_evaluation": None,
            "should_regenerate_queries": None
        }
                        

In [ ]:
state = {
    'assistant_info': {'assistant_type': 'Tour guide assistant', 'assistant_instructions': 'You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.', 'user_question': 'What can I see and do in the Spanish town of Astorga?'},
    'user_question': 'What can I see and do in the Spanish town of Astorga?',
    'iteration_count': 0,    
    'llm': LocalLLM(),
}
result = GenerateSearchQueries(state)
print(result())

In [ ]:
state = {
    'assistant_info': {'assistant_type': 'Tour guide assistant', 'assistant_instructions': 'You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.', 'user_question': 'What can I see and do in the Spanish town of Astorga?'},
    'user_question': 'What can I see and do in the Spanish town of Astorga?',
    'iteration_count': 1,
    'search_queries': [{'search_query': 'Architectural analysis of the Palacio Episcopal de Astorga', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'search_query': "Astorga's strategic importance on the Vía de la Plata Roman road", 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'search_query': 'Local gastronomic traditions and market highlights of Astorga, León', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}],
    'llm': LocalLLM(),
}
result = GenerateSearchQueries(state)
print(result())

In [ ]:
class WebSearches:
    def __init__(self, state: Dict[str, Any]):
        self._llm = state['llm']()
        self._ws = state['ws']
        self._search_queries = state["search_queries"]
        
    def __call__(self) -> Dict[str, Any]:
        """
        Perform web searches based on the generated search queries.
        """
        search_results = []
        print(f"Performing web searches for {len(self._search_queries)} queries...")
        for query_obj in self._search_queries:
            search_query = query_obj['search_query']
            user_question = query_obj['user_question']
            print(f"Searching for: {search_query}")
            urls = self._ws.search(web_query=search_query, num_results=NUM_SEARCH_RESULTS_PER_QUERY)
            for url in urls:
                search_results.append({
                    "result_url": url,
                    "search_query": search_query,
                    "user_question": user_question,
                })

        print(f"Found {len(urls)} results for query: {search_query}")        
        return {"search_results": search_results}


In [ ]:
state = {
    'search_queries': [
        {'search_query': 'Astorga must-see attractions and tourist guide', 'user_question': 'What can I see and do in the Spanish town of Astorga?'},
        {'search_query': 'Astorga historical significance and unique architecture', 'user_question': 'What can I see and do in the Spanish town of Astorga?'},
        {'search_query': 'Things to do and local culture in Astorga, Spain', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}
    ],
    'relevance_evaluation': None,
    'should_regenerate_queries': None,
    'ws': WebSearch(),
    'llm': LocalLLM(),
}
result = WebSearches(state)
print(result())

In [ ]:
class Summarize:
    def __init__(self, state: Dict[str, Any]):
        self._ws = state['ws']
        self._llm = state['llm']()
        self._search_results = state["search_results"]
        self._template = self._read_template()

    def _read_template(self):
        SUMMARY_INSTRUCTIONS = '''
        Read the following text:
        Text: {search_result_text} 
        
        Using the above text, answer in short the following question.
        Question: {search_query}
         
        If you cannot answer the question above using the text provided above, then just summarize the text. 
        Include all factual information, numbers, stats etc if available.
        '''        
        return PromptTemplate.from_template(template=SUMMARY_INSTRUCTIONS)

    def __call__(self) -> Dict[str, Any]:
        summaries = []        
        print(f"Summarizing {len(self._search_results)} search results...")
        
        # For each search result, get the text and summarize it
        for result in self._search_results:
            result_url = result["result_url"]
            search_query = result["search_query"]
            user_question = result["user_question"]
            
            print(f"Scraping content from: {result_url}")
            search_result_text = self._ws.scrape(url=result_url)[:RESULT_TEXT_MAX_CHARACTERS]
            prompt = self._template.format(
                search_result_text=search_result_text,
                search_query=search_query
            )
            
            summary_response = self._llm.invoke(prompt)
            summary = {
                "summary": f"Source Url: {result_url}\nSummary: {summary_response.content}",
                "result_url": result_url,
                "user_question": user_question,
            }
            
            summaries.append(summary)
            print(f"Successfully summarized content from: {result_url}")
        
        research_summary = "\n\n".join([s["summary"] for s in summaries])
        print(f"Created research summary with {len(summaries)} sources")
        return {
            "search_summaries": summaries,
            "research_summary": research_summary,
        }


In [ ]:
state = {
    'search_results': [
        {'result_url': 'https://www.komoot.com/guide/1568259/attractions-around-astorga', 'search_query': 'Astorga must-see attractions and tourist guide', 'user_question': 'What can I see and do in the Spanish town of Astorga?'},
        {'result_url': 'https://en.wikipedia.org/wiki/Ice_cream', 'search_query': 'Ice cream', 'user_question': 'What can I see and do in the Spanish town of Astorga?'},        
    ],
    'ws': WebSearch(),
    'llm': LocalLLM(),
}
result = Summarize(state)
print(result())

In [ ]:
class Evaluate:
    def __init__(self, state: Dict[str, Any]):
        self._llm = state['llm']()
        self._research_summary = state.get('research_summary', '')
        self._prompt = self._read_template().format(
            user_question=state['user_question'],
            research_summary=self._research_summary,
        )
        
    def _read_template(self):
        template = '''
        You are an expert research evaluator. Your task is to evaluate the relevance of search results 
        to the original research question.
        
        Original research question: {user_question}
        
        Search result summaries:
        {research_summary}
        
        For each search result summary, determine if it is relevant to answering the original question.
        Then calculate what percentage of the search results are relevant.
    
        Return the output as a JSON without the ```json header.
        Return your evaluation as a JSON object with the following structure:
        {{
            "relevance_percentage": <percentage of relevant results as a number between 0 and 100>,
            "explanation": <brief explanation of your evaluation>,
            "relevant_count": <number of relevant summaries>,
            "total_count": <total number of summaries>
        }}
        '''
        return PromptTemplate.from_template(template=template)
        
    def __call__(self) -> Dict[str, Any]:
        """
        Evaluate the relevance of search summaries to the original question.
        If less than 50% of summaries are relevant, return to search query generation.
        """        
        print("Evaluating relevance of search summaries to the original question...")
        
        # If there are no summaries, we need to regenerate queries
        if not self._research_summary:
            print("No search summaries found. Regenerating search queries...")
            return {"should_regenerate_queries": True}
            
        evaluation_response = self._llm.invoke(self._prompt)
        evaluation = json.loads(evaluation_response.content)
        relevance_percentage = evaluation.get("relevance_percentage", 0)
            
        # Determine if we should regenerate queries (less than 50% relevant)
        should_regenerate = relevance_percentage < 50
        if should_regenerate:
            print(f"Only {relevance_percentage}% of search results are relevant. Regenerating search queries...")
        else:
            print(f"{relevance_percentage}% of search results are relevant. Proceeding to write research report...")
        
        return {
            "relevance_evaluation": evaluation,
            "should_regenerate_queries": should_regenerate
        }


In [ ]:
state = {
    'llm': LocalLLM(),
    'user_question': 'What can I see and do in the Spanish town of Astorga?',
    'research_summary': 'Source Url: https://www.komoot.com/guide/1568259/attractions-around-astorga\nSummary: Astorga, located in León, Castile and León, Spain, is a significant stop on the Camino de Santiago pilgrimage route, offering a blend of Roman heritage, religious significance, and local culture.\n\n**Must-See Attractions and Historical Sites:**\n\n*   **Cathedral of Saint Mary of Astorga:** A major religious site that can be explored via an audio guide system available in multiple languages.\n*   **Episcopal Palace of Astorga (Gaudí Palace):** A notable neo-Gothic style building designed by the modernist architect Antonio Gaudí.\n*   **Roman Heritage:** Visitors can explore significant sections of the ancient Roman Walls (a testament to its past as Asturica Augusta).\n*   **Roman Museum (Museo Romano / La Ergástula):** Located within an impressive Roman structure, this museum displays artifacts such as mosaics and ceramics.\n*   **Town Hall (Ayuntamiento):** A beautiful 17th-century Baroque building situated in Plaza Mayor.\n*   **Rabanal del Camino — Village and Church:** A settlement in the province of León. The church is of Romanesque origin, built in the 12th century, and is believed to have been constructed by the Knights Templar.\n*   **The Garden of the Soul:** Identified as a rest area on the Camino de Santiago.\n*   **Trail Section:** The segment running between Foncebadón and Rabadal del Camino is approximately 5 km long.\n\n**General Context and Visitor Information:**\n\n*   Astorga is situated on the left bank of the Tuerto River and a spur of the Manzanal mountain chain.\n*   The area offers various outdoor activities, including cycling (Via de la Plata and EuroVelo 3), hiking, running, and MTB trails.\n*   The Komoot app, a navigation tool for outdoor enthusiasts, rates the experience at 4.8/5 stars based on more than 300k user ratings.\n*   The information was last updated on May 10, 2026.\n\nSource Url: https://en.wikipedia.org/wiki/Ice_cream\nSummary: This summary covers the definition, types, and historical development of ice cream, based on the provided text.\n\n***\n\n### 🍦 Summary of Ice Cream\n\n**Definition and Composition**\nIce cream is a frozen dessert typically made from milk or cream. Its core ingredients include a sweetener (sugar or an alternative) and a flavoring, which can be a spice (like cocoa or vanilla) or fruit (like strawberries or peaches). Food coloring and stabilizers may also be added.\n\nThe mixture is cooled below the freezing point of water and stirred to incorporate air spaces, preventing detectable ice crystals. Alternatively, it can be made by whisking a flavored cream base with liquid nitrogen. The resulting texture is a smooth, semi-solid foam that is solid at very low temperatures (below 2°C or 35°F) but becomes more malleable as the temperature rises.\n\n**Uses and Variations**\nIce cream is versatile and can be served in dishes, eaten with a spoon, or licked from edible cones. It can be used as an ingredient in cold dishes (sundaes, milkshakes, ice cream floats) or in baked items (Baked Alaska).\n\n*   **Types:**\n    *   **Gelato:** The Italian version of ice cream.\n    *   **Frozen Custard:** A type of rich ice cream.\n    *   **Soft Serve:** A softer variety often served at amusement parks and fast-food restaurants in the United States.\n    *   **Dairy Alternatives:** Available for those who are lactose intolerant, allergic to dairy, or vegan, using bases like goat\'s, sheep\'s, soy, oat, cashew, coconut, almond milk, or tofu.\n    *   **Fruit-Based:** Banana "nice cream" is noted as a 100% fruit-based vegan alternative.\n    *   **Yogurt-Based:** Frozen yoghurt ("froyo") is similar but uses yogurt and can be lower in fat.\n    *   **Not Ice Cream:** Fruity sorbets or sherbets are distinct from ice creams.\n\n**Regulation and Naming**\nIn countries like the United States and the United Kingdom, the commercial use of the term "ice cream" is regulated by governments, which assess the relative quantities of main ingredients, particularly the amount of butterfat from cream. Products that do not meet these criteria may be labeled "frozen dairy dessert." In contrast, some countries, such as Italy and Argentina, use one word for all variants.\n\n**Historical Development**\n\n*   **Origins:** The origins of frozen desserts are obscure.\n    *   Some sources date the history of ice cream to Persia in **550 BC**.\n    *   A Roman cookbook from the **1st century** included recipes for sweet desserts sprinkled with snow.\n    *   Persian records from the **2nd century** detail sweetened drinks chilled with ice.\n    *   In Japan, **Kakigōri** (ice with flavored syrup) dates back to the Heian period, when blocks of ice were shaved for the aristocracy.\n*   **Early Scientific Process:** The earliest known written process to artificially make ice is found in the **13th-century** writings of Syrian historian Ibn Abi Usaybi\'a. The ability to freeze cream was made easier by the discovery of the **endothermic effect** (the addition of salt lowered the melting point of ice, allowing it to freeze).\n*   **Kulfi (Indian Subcontinent):** In the **16th century**, the Mughal Empire used relays of horsemen to bring ice from the Hindu Kush to Delhi to create *kulfi*, a popular frozen dairy dessert. Although Delhi is often cited as its birthplace, Australian historian Charmaine O\'Brien suggests it likely originated in Persia or Samarkand.\n*   **Europe:** The technique of freezing was unknown in Europe prior to the **16th century**.\n    *   By the latter part of the **17th century**, sorbets and ice creams were made using the refrigerant effect.\n    *   **Legendary Introduction:** A legend credits Catherine de\' Medici with bringing flavored sorbet ices to France upon marrying the Duke of Orléans in **1533**.\n    *   **French Developments:**\n        *   **1665:** The *Catalogue des Marchandises rares...* listed a frozen sorbet, consumed using a container plunged into ice and saltpetre.\n        *   The practice of cooling drinks with ice and snow emerged in Paris during the **16th century**.\n        *   **1682:** *Le Nouveau confiturier françois* provided a recipe for "neige de fleur d\'orange."\n        *   **1686:** Italian Francesco dei Coltelli opened an ice cream café in Paris.',
}
result = Evaluate(state)
print(result())

In [ ]:
class WriteReport:
    def __init__(self, state: Dict[str, Any]):
        self._llm = state['llm']()
        self._prompt = self._read_template().format(
            research_summary=state.get('research_summary', ''),
            user_question=state['user_question']
        )        
        
    def _read_template(self):
        template = '''
        You are an AI critical thinker research assistant. Your sole purpose is to write well written, critically acclaimed, objective and structured reports on given text.        
        Information: 
        --------
        {research_summary}
        --------
        
        Using the above information, answer the following question or topic: "{user_question}" in a detailed report -- \
        The report should focus on the answer to the question, should be well structured, informative, \
        in depth, with facts and numbers if available and a minimum of 1,200 words.
        
        You should strive to write the report as long as you can using all relevant and necessary information provided.
        You must write the report with markdown syntax.
        You MUST determine your own concrete and valid opinion based on the given information. Do NOT deter to general and meaningless conclusions.
        Write all used source urls at the end of the report, and make sure to not add duplicated sources, but only one reference for each.
        You must write the report in apa format.
        '''
        return PromptTemplate.from_template(template=template)

    def __call__(self) -> Dict[str, Any]:
        """Write a research report based on the summarized search results."""            
        response = self._llm.invoke(self._prompt)
        return {"final_report": response.content}


In [ ]:
class ResearchGraph:
    def __init__(self, state_schema: type):
        self.state_schema = state_schema
        self.workflow = self._create_workflow()

    def _create_workflow(self) -> StateGraph:
        workflow = StateGraph(self.state_schema)

        # Add Nodes - Mapping to internal methods
        workflow.add_node("select_assistant", self.select_assistant)
        workflow.add_node("generate_search_queries", self.generate_search_queries)
        workflow.add_node("perform_web_searches", self.perform_web_searches)
        workflow.add_node("summarize_search_results", self.summarize_search_results)
        workflow.add_node("evaluate_search_relevance", self.evaluate_search_relevance)
        workflow.add_node("write_research_report", self.write_research_report)

        # Static Edges
        workflow.set_entry_point("select_assistant")
        workflow.add_edge("select_assistant", "generate_search_queries")
        workflow.add_edge("generate_search_queries", "perform_web_searches")
        workflow.add_edge("perform_web_searches", "summarize_search_results")
        workflow.add_edge("summarize_search_results", "evaluate_search_relevance")

        # Conditional Edges
        workflow.add_conditional_edges(
            "evaluate_search_relevance",
            self.route_based_on_relevance,
            {
                "generate_search_queries": "generate_search_queries",
                "write_research_report": "write_research_report"
            }
        )
        workflow.add_edge("write_research_report", END)

        return workflow

    # --- Node Methods ---
    def select_assistant(self, state: Dict[str, Any]) -> Dict[str, Any]:
        return Assistant(state)()

    def generate_search_queries(self, state: Dict[str, Any]) -> Dict[str, Any]:
        return GenerateSearchQueries(state)()

    def perform_web_searches(self, state: Dict[str, Any]) -> Dict[str, Any]:
        return WebSearches(state)()

    def summarize_search_results(self, state: Dict[str, Any]) -> Dict[str, Any]:
        return Summarize(state)()

    def evaluate_search_relevance(self, state: Dict[str, Any]) -> Dict[str, Any]:
        return Evaluate(state)()

    def write_research_report(self, state: Dict[str, Any]) -> Dict[str, Any]:
        return WriteReport(state)()

    # --- Routing Logic ---
    def route_based_on_relevance(self, state: Dict[str, Any]) -> str:
        iteration_count = state.get("iteration_count", 0) + 1
        state["iteration_count"] = iteration_count # Note: Usually handled via Reducers in LangGraph
        
        if iteration_count >= 3:
            return "write_research_report"
        
        if state.get("should_regenerate_queries", False):
            return "generate_search_queries"
        
        return "write_research_report"

    def compile(self, checkpointer=None):
        return self.workflow.compile(checkpointer=checkpointer)

In [ ]:
question = "Describe the Airavatesvara temple in India."
initial_state = {
    "llm": LocalLLM(),
    "ws": WebSearch(),
    "user_question": question,
    "assistant_info": None,
    "search_queries": None,
    "search_results": None,
    "search_summaries": None,
    "research_summary": None,
    "final_report": None,
    "relevance_evaluation": None,
    "should_regenerate_queries": None,
    "iteration_count": 0,
}

research_app = ResearchGraph(ResearchState)
app = research_app.compile()
result = app.invoke(initial_state)

In [ ]:
print(result['final_report'])